## **2025-05-29-Create-GPF-Library-OSW**

In this notebook create the GPF library. iRT files are created here as well.

In [1]:
import sys
import os
sys.path.append("../../src/")
import createGPFLibrary
import pandas as pd
import polars as pl
from matplotlib_venn import venn2

#### **Create GPF Library**

Create GPF library using the XGB and the LDA scripts

In [2]:
gpfLib_xgb = createGPFLibrary.run("osw/pyprophet_XGB/merged.parquet")

Reading parquet
Complete parquet loading
Normalizing Intensities ... 


**Note:** XGBoost distributions look reasonable.

In [3]:
gpfLib_lda = createGPFLibrary.run("osw/pyprophet_LDA/merged.parquet")

Reading parquet
Complete parquet loading
Normalizing Intensities ... 


**Note:** LDA Distribution is reasonable enough for MS1-MS2 Level

#### **XGBoost Library is slightly bigger than LDA library**

In [4]:
gpfLib_lda = gpfLib_lda.sort_values(by=['ModifiedPeptideSequence', 'PrecursorCharge'])
gpfLib_lda['Precursor'] = gpfLib_lda['ModifiedPeptideSequence'] + gpfLib_lda['PrecursorCharge'].astype(str)
gpfLib_lda['Precursor'] = gpfLib_lda['Precursor'].str.replace(r'^\.', '', regex=True)

gpfLib_lda['Precursor'].drop_duplicates()

22344682     AAAAAAAAAPAAAATAPTTAATTAATAAQ2
19943002         AAAAAAAAVPSAGPAGPAPTSAAGR2
1401142                        AAAAAAALQAK2
16746011    AAAAAATAPPSPGPAQ(UniMod:7)PGPR2
16664579              AAAAAATAPPSPGPAQPGPR2
                         ...               
8991555               YYYQGC(UniMod:4)ASWK2
13813881                      YYYSDNFFDGQR2
15777621                     YYYVPADFVEYEK2
11448303                       YYYWAVNPQDR2
1284564                             YYYYHR2
Name: Precursor, Length: 139136, dtype: object

In [5]:
gpfLib_xgb = gpfLib_xgb.sort_values(by=['ModifiedPeptideSequence', 'PrecursorCharge'])
gpfLib_xgb['Precursor'] = gpfLib_xgb['ModifiedPeptideSequence'] + gpfLib_xgb['PrecursorCharge'].astype(str)
gpfLib_xgb['Precursor'] = gpfLib_xgb['Precursor'].str.replace(r'^\.', '', regex=True)

gpfLib_xgb['Precursor'].drop_duplicates()

22344681     AAAAAAAAAPAAAATAPTTAATTAATAAQ2
19943007         AAAAAAAAVPSAGPAGPAPTSAAGR2
1401137                        AAAAAAALQAK2
11690238                   AAAAAAALQAKSDEK2
16746006    AAAAAATAPPSPGPAQ(UniMod:7)PGPR2
                         ...               
8991560               YYYQGC(UniMod:4)ASWK2
13813886                      YYYSDNFFDGQR2
15777623                     YYYVPADFVEYEK2
11448303                       YYYWAVNPQDR2
1284569                             YYYYHR2
Name: Precursor, Length: 156638, dtype: object

We see that the numbers here are considerably more than what could achieve when refine the experimental library.

**Update June 5th:** We see the numbers are slightly higher than earlier as well.

#### **Export Library**

In [6]:
gpfLib_xgb.to_csv("osw/2025-06-05-Refine-BrukerLib-With-GPF-OSW.tsv", sep='\t')

#### **Create and Export iRT files**

To create the iRT files, just take the original iRT files and replace the RT and IM values with the ones from this new library.

In [7]:
linIrt = pd.read_csv("../K562-Bruker-Library-Analysis/formatted/2025-05-23-linear-irt-in-silico.tsv", sep='\t')
nonLinIrt = pd.read_csv("../K562-Bruker-Library-Analysis/formatted/2025-03-06-nonlinear-irt-in-silico.tsv", sep='\t')
linIrt['Precursor'] = linIrt['ModifiedPeptideSequence'] + linIrt['PrecursorCharge'].astype(str)
linIrt['Precursor'] = linIrt['Precursor'].str.replace(r'^\.', '', regex=True)
nonLinIrt['Precursor'] = nonLinIrt['ModifiedPeptideSequence'] + nonLinIrt['PrecursorCharge'].astype(str)

nonLinIrt['Precursor'] = nonLinIrt['Precursor'].str.replace(r'^\.', '', regex=True)

In [8]:
linIrt = gpfLib_xgb[gpfLib_xgb['Precursor'].isin(linIrt['Precursor'])]
nonLinIrt = gpfLib_xgb[gpfLib_xgb['Precursor'].isin(nonLinIrt['Precursor'])]

In [9]:
nonLinIrt['Precursor'].drop_duplicates()

13510943                  AAAAALSQQQSLQER2
14821901                AAAAAWEEPSSGNGTAR2
17324766    AAAAGLGHPASPGGSEDGPPGSEEEDAAR3
14656460                AAAAPFQTSQASASAPR2
15635903                  AAAEDVNVTFEDQQK2
                         ...              
3078335                     YYQTIGNHASYYK3
8929567                        YYSFFDLDPK2
7657456                        YYSLDELSEK2
1248025                          YYVLNALK2
5097514                          YYYIPQYK2
Name: Precursor, Length: 15443, dtype: object

Have 14K peptide precursors in the non linear iRT which should be sufficient. 

In [10]:
linIrt

,PrecursorMz,ProductMz,ProteinId,ModifiedPeptideSequence,PeptideSequence,PrecursorCharge,PeptideGroupLabel,NormalizedRetentionTime,PrecursorIonMobility,LibraryIntensity,Annotation,Precursor
13510943,786.4104,1103.5440,Q8WUQ7,AAAAALSQQQSLQER,AAAAALSQQQSLQER,2,AAAAALSQQQSLQER_2,36.258174,1.063846,10000.000000,y9^1,AAAAALSQQQSLQER2
13510942,786.4104,1016.5120,Q8WUQ7,AAAAALSQQQSLQER,AAAAALSQQQSLQER,2,AAAAALSQQQSLQER_2,36.258174,1.063846,4054.273372,y8^1,AAAAALSQQQSLQER2
13510941,786.4104,888.4534,Q8WUQ7,AAAAALSQQQSLQER,AAAAALSQQQSLQER,2,AAAAALSQQQSLQER_2,36.258174,1.063846,6499.537180,y7^1,AAAAALSQQQSLQER2
13510940,786.4104,760.3948,Q8WUQ7,AAAAALSQQQSLQER,AAAAALSQQQSLQER,2,AAAAALSQQQSLQER_2,36.258174,1.063846,3313.792039,y6^1,AAAAALSQQQSLQER2
13510939,786.4104,632.3362,Q8WUQ7,AAAAALSQQQSLQER,AAAAALSQQQSLQER,2,AAAAALSQQQSLQER_2,36.258174,1.063846,5880.900956,y5^1,AAAAALSQQQSLQER2
...,...,...,...,...,...,...,...,...,...,...,...,...
6548891,583.8062,937.4989,Q9Y5B9,AASITSEVFNK,AASITSEVFNK,2,AASITSEVFNK_2,54.605259,0.913226,1378.575235,y8^1,AASITSEVFNK2
6548890,583.8062,824.4149,Q9Y5B9,AASITSEVFNK,AASITSEVFNK,2,AASITSEVFNK_2,54.605259,0.913226,10000.000000,y7^1,AASITSEVFNK2
6548889,583.8062,723.3672,Q9Y5B9,AASITSEVFNK,AASITSEVFNK,2,AASITSEVFNK_2,54.605259,0.913226,4173.624682,y6^1,AASITSEVFNK2
6548888,583.8062,636.3352,Q9Y5B9,AASITSEVFNK,AASITSEVFNK,2,AASITSEVFNK_2,54.605259,0.913226,1215.372772,y5^1,AASITSEVFNK2


In [11]:
len(linIrt['Precursor'].drop_duplicates())

39

39 peptide precursors for linear iRT should be reasonable.

In [12]:
linIrt.to_csv("osw/2025-05-23-linear-irt-Bruker-GPF.tsv", sep='\t', index=False)
nonLinIrt.to_csv("osw/2025-03-06-nonlinear-irt-Bruker-GPF.tsv", sep='\t', index=False)

In [12]:
linIrt.to_csv("osw/2025-05-23-linear-irt-Bruker-GPF.tsv", sep='\t', index=False)
nonLinIrt.to_csv("osw/2025-03-06-nonlinear-irt-Bruker-GPF.tsv", sep='\t', index=False)